# Lecture 17: NLP in Industry Domains
### NLP Course 2027

---

## Learning Outcomes
- Preprocess and analyze social media text
- Apply aspect-based sentiment analysis to e-commerce reviews
- Use domain-specific NLP for healthcare and legal text
- Understand the unique challenges of each domain

**Primary Reference:** *Practical NLP* Ch.8, 9, 10

## 1. Social Media NLP

### Challenges
- Abbreviations: `lol`, `omg`, `imo`, `tbh`, `afaik`
- Hashtags: `#NLP` - meaningful signal, needs splitting
- Mentions: `@user` - entity or noise
- Emojis: 😊 => positive signal
- URLs: usually noise
- Character repetition: `sooooo good`
- Code-switching: mixing languages
- No grammar, no punctuation

### Applications
- Sentiment analysis (brand monitoring)
- Trend detection (Twitter trends)
- Crisis detection (public health)
- Hate speech and toxicity detection
- Political discourse analysis

In [1]:
import re

def preprocess_tweet(tweet, keep_hashtags=True):
    """Social media text preprocessor."""
    # Remove URLs
    tweet = re.sub(r'https?://\S+|www\.\S+', '', tweet)
    # Handle hashtags: keep the word
    if keep_hashtags:
        tweet = re.sub(r'#(\w+)', r'\1', tweet)
    else:
        tweet = re.sub(r'#\w+', '', tweet)
    # Remove mentions
    tweet = re.sub(r'@\w+', '', tweet)
    # Remove emoji (basic: strip non-ASCII)
    tweet = tweet.encode('ascii', 'ignore').decode('ascii')
    # Normalize repeated characters: sooooo -> sooo
    tweet = re.sub(r'(.)\1{2,}', r'\1\1', tweet)
    # Expand abbreviations
    abbrevs = {
        r'\bu\b': 'you', r'\br\b': 'are', r'\bur\b': 'your',
        r'\bgr8\b': 'great', r'\bthx\b': 'thanks',
        r'\bbtw\b': 'by the way', r'\bimo\b': 'in my opinion',
        r'\blol\b': 'laugh out loud', r'\bomg\b': 'oh my god',
        r'\bbrb\b': 'be right back', r'\bimo\b': 'in my opinion',
    }
    for pattern, expansion in abbrevs.items():
        tweet = re.sub(pattern, expansion, tweet, flags=re.IGNORECASE)
    # Normalize whitespace
    tweet = re.sub(r'\s+', ' ', tweet).strip()
    return tweet.lower()

tweets = [
    '@JohnDoe gr8 job on #NLP! Can not believe u did it lol https://t.co/xyz',
    '#AI is taking over omg! imo we r all doomed btw thx for the update!!!',
    'This product is sooooo amazing!!! @company #bestpurchase ever',
]
for tweet in tweets:
    print(f'Original:  {tweet}')
    print(f'Processed: {preprocess_tweet(tweet)}')
    print()

Original:  @JohnDoe gr8 job on #NLP! Can not believe u did it lol https://t.co/xyz
Processed: great job on nlp! can not believe you did it laugh out loud

Original:  #AI is taking over omg! imo we r all doomed btw thx for the update!!!
Processed: ai is taking over oh my god! in my opinion we are all doomed by the way thanks for the update!!

Original:  This product is sooooo amazing!!! @company #bestpurchase ever
Processed: this product is soo amazing!! bestpurchase ever



In [2]:
from transformers import pipeline

# Sentiment analysis on tweets
sentiment = pipeline('text-classification',
                     model='cardiffnlp/twitter-roberta-base-sentiment-latest')

tweet_samples = [
    'Just got the new iPhone and it is AMAZING! Best phone ever! #Apple',
    'The customer service at this airline is absolutely terrible. 3 hour delay with zero communication.',
    'Meh, the food was okay but nothing special. Probably wont go back.',
    'Cannot believe how good this NLP course is. Highly recommend!',
    'This software is so slow and buggy. Complete waste of money.',
]
print('Tweet Sentiment Analysis:')
for tweet in tweet_samples:
    result = sentiment(tweet)[0]
    print(f'  {result["label"]:10s} ({result["score"]:.3f}) | {tweet[:60]}')

/opt/miniconda3/envs/nlp-course/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Hardware accelerato

Tweet Sentiment Analysis:
  positive   (0.990) | Just got the new iPhone and it is AMAZING! Best phone ever! 
  negative   (0.961) | The customer service at this airline is absolutely terrible.
  negative   (0.852) | Meh, the food was okay but nothing special. Probably wont go
  positive   (0.988) | Cannot believe how good this NLP course is. Highly recommend
  negative   (0.956) | This software is so slow and buggy. Complete waste of money.


## 2. E-Commerce Text Analytics

### Key Tasks
- **Review sentiment**: positive/negative/neutral
- **Aspect-based sentiment analysis (ABSA)**: sentiment per feature
- **Product classification**: assign to category
- **Search query understanding**: parse user intent

### Aspect-Based Sentiment Analysis
```
Review: 'The camera quality is excellent but the battery life is terrible.'

Aspect        Sentiment
-----------   ---------
camera        positive
battery life  negative
```

In [3]:
# Rule-based ABSA using dependency parsing
try:
    import spacy
    nlp = spacy.load('en_core_web_sm')
    HAS_SPACY = True
except:
    HAS_SPACY = False
    print('spaCy not available. Install: pip install spacy && python -m spacy download en_core_web_sm')

# Lexicon-based ABSA
POS_WORDS = {'great', 'excellent', 'amazing', 'good', 'fantastic', 'love', 'perfect',
             'best', 'wonderful', 'outstanding', 'brilliant', 'fast', 'easy'}
NEG_WORDS = {'terrible', 'bad', 'awful', 'poor', 'slow', 'broken', 'disappointing',
             'horrible', 'worst', 'useless', 'difficult', 'expensive', 'cheap'}

ASPECT_KEYWORDS = {
    'battery': ['battery', 'battery life', 'charge', 'charging'],
    'camera': ['camera', 'photo', 'picture', 'image', 'video', 'lens'],
    'screen': ['screen', 'display', 'brightness', 'resolution'],
    'performance': ['performance', 'speed', 'fast', 'slow', 'processor', 'lag'],
    'design': ['design', 'build', 'look', 'feel', 'weight', 'size'],
    'price': ['price', 'cost', 'value', 'expensive', 'cheap', 'worth'],
}

def simple_absa(review):
    """Simple lexicon-based aspect sentiment analysis."""
    review_lower = review.lower()
    results = {}
    words = set(review_lower.split())
    for aspect, keywords in ASPECT_KEYWORDS.items():
        if any(kw in review_lower for kw in keywords):
            pos = sum(1 for w in words if w in POS_WORDS)
            neg = sum(1 for w in words if w in NEG_WORDS)
            if pos > neg:
                results[aspect] = 'positive'
            elif neg > pos:
                results[aspect] = 'negative'
            else:
                results[aspect] = 'neutral'
    return results

reviews = [
    'The camera quality is excellent but the battery life is terrible.',
    'Fast performance and great design. The price is a bit expensive though.',
    'The screen is amazing and the camera takes wonderful photos. Battery lasts all day!',
]
for review in reviews:
    aspects = simple_absa(review)
    print(f'Review: {review}')
    print(f'Aspects: {aspects}')
    print()

spaCy not available. Install: pip install spacy && python -m spacy download en_core_web_sm
Review: The camera quality is excellent but the battery life is terrible.
Aspects: {'battery': 'positive', 'camera': 'positive'}

Review: Fast performance and great design. The price is a bit expensive though.
Aspects: {'performance': 'positive', 'design': 'positive', 'price': 'positive'}

Review: The screen is amazing and the camera takes wonderful photos. Battery lasts all day!
Aspects: {'battery': 'positive', 'camera': 'positive', 'screen': 'positive'}



In [4]:
# Product review classification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Simulated labeled reviews
train_reviews = [
    ('The battery lasts 2 days and charges quickly', 'battery'),
    ('Battery died after 6 months, very disappointing', 'battery'),
    ('Camera takes stunning 4K photos with great detail', 'camera'),
    ('Photos are blurry and washed out in low light', 'camera'),
    ('Lightning fast processor, apps open instantly', 'performance'),
    ('Constant lag and crashes make this unusable', 'performance'),
    ('Slim design with premium materials and solid build', 'design'),
    ('The display is bright and colors are vivid', 'screen'),
    ('The screen cracks easily, very fragile glass', 'screen'),
    ('Excellent value for money at this price point', 'price'),
    ('Way too expensive for what you get', 'price'),
]
texts, labels = zip(*train_reviews)
cv = TfidfVectorizer(ngram_range=(1, 2))
clf = Pipeline([('tfidf', cv), ('lr', LogisticRegression(max_iter=200))])
clf.fit(texts, labels)

new_reviews = [
    'I love the beautiful AMOLED display on this phone',
    'The 5000mAh battery easily lasts 3 days',
    'This device is just too heavy and bulky',
    'Snapdragon processor handles all games without a sweat',
]
for r in new_reviews:
    pred = clf.predict([r])[0]
    print(f'[{pred:12s}] {r}')

[screen      ] I love the beautiful AMOLED display on this phone
[battery     ] The 5000mAh battery easily lasts 3 days
[price       ] This device is just too heavy and bulky
[performance ] Snapdragon processor handles all games without a sweat


## 3. Healthcare NLP

### Challenges in Clinical NLP
- **Abbreviations**: MI (myocardial infarction), SOB (shortness of breath), Dx (diagnosis)
- **Negation**: 'No chest pain' - detecting negative is critical
- **Medical jargon**: technical terms not in general NLP models
- **Privacy**: HIPAA compliance, de-identification required
- **Irregular text**: clinical notes are often telegraphic

### Tools
- **SciSpaCy**: spaCy models trained on biomedical text
- **ClinicalBERT** / **BioBERT**: domain-adapted BERT models
- **MetaMap**: map text to UMLS medical concepts

In [5]:
# Medical text preprocessing and NER concept
# !pip install scispacy
# !pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.1/en_core_sci_sm-0.5.1.tar.gz

try:
    import scispacy
    import spacy
    nlp_med = spacy.load('en_core_sci_sm')
    text = 'Patient presents with acute MI and hypertension. No SOB. Administer aspirin 325mg stat.'
    doc = nlp_med(text)
    print('Medical Entities:')
    for ent in doc.ents:
        print(f'  [{ent.label_:10s}] {ent.text}')
except ImportError:
    print('SciSpaCy not installed. Using BioBERT NER pipeline as alternative.')
    from transformers import pipeline
    bio_ner = pipeline('ner', model='allenai/scibert_scivocab_uncased',
                       aggregation_strategy='simple')
    text = 'Patient presents with acute myocardial infarction and hypertension.'
    print(f'Text: {text}')
    print('Entities (SciBERT):')
    for ent in bio_ner(text):
        print(f'  [{ent["entity_group"]:10s}] {ent["word"]}')

Medical Entities:
  [ENTITY    ] Patient
  [ENTITY    ] acute MI
  [ENTITY    ] hypertension
  [ENTITY    ] No SOB
  [ENTITY    ] Administer
  [ENTITY    ] aspirin
  [ENTITY    ] stat


In [6]:
# Negation detection: critical for clinical NLP
def detect_negation(sentence):
    """
    Simple rule-based negation detection.
    Returns list of (term, is_negated) tuples.
    """
    NEG_CUES = ['no', 'not', 'without', 'absent', 'denies', 'negative for',
                'ruled out', 'no evidence of', 'unable to']
    ENTITIES = ['fever', 'chest pain', 'shortness of breath', 'cough',
                'hypertension', 'diabetes', 'infection', 'bleeding']
    sentence_lower = sentence.lower()
    results = []
    for entity in ENTITIES:
        if entity in sentence_lower:
            entity_pos = sentence_lower.find(entity)
            # Check for negation cue in 5 words before the entity
            window = sentence_lower[max(0, entity_pos-50):entity_pos]
            is_negated = any(cue in window for cue in NEG_CUES)
            results.append((entity, is_negated))
    return results

clinical_notes = [
    'Patient has no fever and no chest pain. Denies shortness of breath.',
    'Positive for hypertension and diabetes. No evidence of infection.',
    'Patient presents with cough and shortness of breath.',
]
for note in clinical_notes:
    print(f'Note: {note}')
    negations = detect_negation(note)
    for entity, negated in negations:
        status = 'ABSENT' if negated else 'PRESENT'
        print(f'  {entity:25s}: {status}')
    print()

Note: Patient has no fever and no chest pain. Denies shortness of breath.
  fever                    : ABSENT
  chest pain               : ABSENT
  shortness of breath      : ABSENT

Note: Positive for hypertension and diabetes. No evidence of infection.
  hypertension             : PRESENT
  diabetes                 : PRESENT
  infection                : ABSENT

Note: Patient presents with cough and shortness of breath.
  shortness of breath      : PRESENT
  cough                    : PRESENT



## 4. Legal Text NLP

### Challenges
- Long documents (contracts can be 100+ pages)
- Technical legal vocabulary and Latin terms
- Precision matters: a single word can change meaning entirely
- Citation structures: case law references
- Dated language in historical documents

### Key Tasks
- Contract clause classification
- Legal NER (parties, dates, obligations)
- Document similarity for case law retrieval
- Risk clause detection

In [7]:
# Legal clause classification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Sample legal clause training data
clause_data = [
    ('This agreement shall be governed by the laws of California', 'governing_law'),
    ('The laws of New York shall apply to this contract', 'governing_law'),
    ('Either party may terminate this agreement with 30 days written notice', 'termination'),
    ('This agreement may be terminated immediately upon material breach', 'termination'),
    ('Company shall indemnify and hold harmless the other party', 'indemnification'),
    ('Each party indemnifies the other against third-party claims', 'indemnification'),
    ('All disputes shall be resolved through binding arbitration', 'dispute_resolution'),
    ('Any dispute shall be submitted to the courts of San Francisco', 'dispute_resolution'),
    ('Payment is due within 30 days of invoice date', 'payment_terms'),
    ('The license fee of $10,000 is payable annually in advance', 'payment_terms'),
    ('This agreement is confidential and shall not be disclosed', 'confidentiality'),
    ('The parties agree to keep all terms of this agreement confidential', 'confidentiality'),
]
texts, labels = zip(*clause_data)
cfr = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,3))),
                ('lr', LogisticRegression(max_iter=200))])
cfr.fit(texts, labels)

# Test on new clauses
new_clauses = [
    'This contract shall be construed under Delaware law.',
    'Either party may terminate upon 60 days prior written notice.',
    'All confidential information shall remain strictly private.',
    'Disputes shall first be mediated and then arbitrated.',
]
for clause in new_clauses:
    pred = cfr.predict([clause])[0]
    prob = max(cfr.predict_proba([clause])[0])
    print(f'[{pred:22s}] ({prob:.2f}) {clause}')

[governing_law         ] (0.20) This contract shall be construed under Delaware law.
[termination           ] (0.27) Either party may terminate upon 60 days prior written notice.
[confidentiality       ] (0.20) All confidential information shall remain strictly private.
[dispute_resolution    ] (0.21) Disputes shall first be mediated and then arbitrated.


## Practice Exercises

See **`Lecture-17-Homework.ipynb`** for the practice exercises accompanying this lecture.

## Summary

| Domain | Key Challenge | NLP Solution |
|--------|---------------|-------------|
| Social Media | Noise, abbreviations | TweetTokenizer, domain models |
| E-commerce | Aspect sentiment | ABSA, review classification |
| Healthcare | Abbreviations, negation | BioBERT, SciSpaCy, negation detection |
| Legal | Long docs, jargon | LegalBERT, clause classification |

**Next Lecture**: Productionizing NLP Systems.

---
*Book reference: Practical NLP Ch.8, 9, 10*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**